# LLM2-Lab — Execução e Pesquisa no Kaggle (NVIDIA Tesla T4 x2 / 30GB VRAM)

Notebook oficial para validação e treino do **LLM2-Lab (Pesquisa IA 2)** no Kaggle Notebooks.

### Características do Ambiente Kaggle:
- **Hardware:** 2x NVIDIA Tesla T4 (~15-16 GiB VRAM cada, totalizando ~30 GiB).
- **Arquitetura Turing:** GPUs T4 não possuem suporte nativo a Bfloat16 (`bf16`). Por isso, este fluxo utiliza **`precision: fp32`** com aceleração completa via **`device: cuda`**, garantindo estabilidade numérica exata e aproveitando os Tensor Cores via SDPA nativo do PyTorch.
- **Arquitetura Testada:** LRM (*Latent Recursive Model*) com passos contínuos de refinamento latente e objetivo preditivo JEPA anti-colapso.


In [ ]:
# =============================================================================
# 1. CHECAGEM DE HARDWARE E DIAGNÓSTICO DE VRAM (KAGGLE T4 x2)
# =============================================================================
import os
import sys
import subprocess
from pathlib import Path
import torch

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
assert torch.cuda.is_available(), 'Ative a aceleração de GPU nas configurações do Kaggle (Settings -> Accelerator -> GPU T4 x2).'

n_gpus = torch.cuda.device_count()
total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpus)) / (1024**3)

print(f'✅ GPUs Detectadas: {n_gpus}')
for i in range(n_gpus):
    prop = torch.cuda.get_device_properties(i)
    print(f'   - GPU {i}: {prop.name} | VRAM: {prop.total_memory / (1024**3):.2f} GiB')
print(f'📦 VRAM Total Disponível: {total_vram_gb:.2f} GiB')
print(f'⚙️ Versão PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')
print(f'💡 Política de Precisão: FP32 nativo para GPU Turing T4 (estabilidade numérica exata).')


In [ ]:
# =============================================================================
# 2. CLONAGEM DO REPOSITÓRIO E INSTALAÇÃO
# =============================================================================
repo_dir = Path('-LLM-2-')
if not repo_dir.exists():
    print('Clonando repositório PedroGGC/-LLM-2-...')
    subprocess.run(['git', 'clone', 'https://github.com/PedroGGC/-LLM-2-.git'], check=True)

os.chdir(repo_dir)
print('Diretório atual:', os.getcwd())

print('Instalando pacote llm2 em modo editável com dependências de pesquisa...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,research]'], check=True)
print('✅ Instalação concluída com sucesso.')


In [ ]:
# =============================================================================
# 3. SUÍTE DE TESTES UNITÁRIOS DE INTEGRIDADE
# =============================================================================
print('Executando suíte de testes de sanidade...')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
print('✅ Todos os testes foram aprovados no Kaggle.')


In [ ]:
# =============================================================================
# 4. PREPARAÇÃO DO CORPUS E TOKENIZAÇÃO OFFLINE
# =============================================================================
print('Gerando corpus de demonstração...')
subprocess.run([sys.executable, 'scripts/make_demo_corpus.py'], check=True)

data_dir = Path('data/processed-byte')
if not data_dir.exists():
    print('Preparando dataset determinístico...')
    subprocess.run([
        sys.executable, '-m', 'llm2', 'prepare',
        '--input', 'data/raw/demo.txt',
        '--output', 'data/processed-byte',
        '--tokenizer', 'byte',
        '--seed', '42'
    ], check=True)
    print('✅ Dataset preparado em data/processed-byte.')
else:
    print('Dataset já existente em data/processed-byte.')


In [ ]:
# =============================================================================
# 5. INSPEÇÃO DA ARQUITETURA E ESTIMATIVA DE FLOPs
# =============================================================================
subprocess.run([
    sys.executable, '-m', 'llm2', 'inspect',
    '--config', 'configs/kaggle_t4.yaml',
    '--meta'
], check=True)


In [ ]:
# =============================================================================
# 6. TREINAMENTO NA GPU (LRM + JEPA EM CUDA FP32)
# =============================================================================
run_dir = Path('runs/kaggle-t4-lrm')
print(f'Iniciando treino com checkpointing determinístico em {run_dir}...')
subprocess.run([
    sys.executable, '-m', 'llm2', 'train',
    '--config', 'configs/kaggle_t4.yaml',
    '--output-dir', str(run_dir)
], check=True)
print('✅ Treinamento e salvamento de checkpoints concluídos.')


In [ ]:
# =============================================================================
# 7. AVALIAÇÃO, RACIOCÍNIO LATENTE E GERAÇÃO
# =============================================================================
latest_ckpt = run_dir / 'checkpoints' / 'step-00000050'

print('--- 1. Avaliação de Perda no Conjunto de Validação ---')
subprocess.run([
    sys.executable, '-m', 'llm2', 'evaluate',
    '--checkpoint', str(latest_ckpt),
    '--device', 'cuda',
    '--precision', 'fp32'
], check=True)

print('\n--- 2. Teste de Raciocínio em Estados Latentes (LRM) ---')
subprocess.run([
    sys.executable, '-m', 'llm2', 'reasoning',
    '--checkpoint', str(latest_ckpt),
    '--device', 'cuda',
    '--precision', 'fp32',
    '--examples', '2',
    '--max-new-tokens', '16',
    '--output', str(run_dir / 'reasoning.json')
], check=True)

print('\n--- 3. Geração Autoregressiva de Texto ---')
subprocess.run([
    sys.executable, '-m', 'llm2', 'generate',
    '--checkpoint', str(latest_ckpt),
    '--prompt', 'O modelo neural',
    '--seed', '42',
    '--max-new-tokens', '30',
    '--device', 'cuda',
    '--precision', 'fp32'
], check=True)
print('\n🎉 Experimento no Kaggle T4 concluído com sucesso!')
